# Análisis exploratorio airports.csv

In [4]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("eda_openflights_airports")
    .enableHiveSupport()
    .getOrCreate()
)

RAW_PATH = "/Obligatorio/landing/openflights/airports.csv"

# Definimos esquema explícito: así documentamos el tipo que DEBERÍA tener cada
# columna y no dependemos de la inferencia automática de Spark.
airports_schema = T.StructType([
    T.StructField("airport_id",  T.IntegerType(), True),
    T.StructField("name",        T.StringType(),  True),
    T.StructField("city",        T.StringType(),  True),
    T.StructField("country",     T.StringType(),  True),
    T.StructField("iata",        T.StringType(),  True),
    T.StructField("icao",        T.StringType(),  True),
    T.StructField("latitude",    T.DoubleType(),  True),
    T.StructField("longitude",   T.DoubleType(),  True),
    T.StructField("altitude",    T.IntegerType(), True),
    T.StructField("timezone",    T.DoubleType(),  True),
    T.StructField("dst",         T.StringType(),  True),
    T.StructField("tz_database", T.StringType(),  True),
    T.StructField("type",        T.StringType(),  True),
    T.StructField("source",      T.StringType(),  True),
])

airports_raw = (
    spark.read
    .option("header", True)
    .schema(airports_schema)
    .csv(RAW_PATH)
)

In [5]:
print("Filas:", airports_raw.count())
print("Columnas:", len(airports_raw.columns))
print("Nombres de columnas:", airports_raw.columns)

airports_raw.printSchema()
airports_raw.show(5, truncate=False)

[Stage 0:>                                                          (0 + 1) / 1]

Filas: 12668
Columnas: 14
Nombres de columnas: ['airport_id', 'name', 'city', 'country', 'iata', 'icao', 'latitude', 'longitude', 'altitude', 'timezone', 'dst', 'tz_database', 'type', 'source']
root
 |-- airport_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- iata: string (nullable = true)
 |-- icao: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- altitude: integer (nullable = true)
 |-- timezone: double (nullable = true)
 |-- dst: string (nullable = true)
 |-- tz_database: string (nullable = true)
 |-- type: string (nullable = true)
 |-- source: string (nullable = true)



+----------+-------------------------------------------+------------+----------------+----+----+------------------+------------------+--------+--------+---+--------------------+-------+-----------+
|airport_id|name                                       |city        |country         |iata|icao|latitude          |longitude         |altitude|timezone|dst|tz_database         |type   |source     |
+----------+-------------------------------------------+------------+----------------+----+----+------------------+------------------+--------+--------+---+--------------------+-------+-----------+
|1         |Goroka Airport                             |Goroka      |Papua New Guinea|GKA |AYGA|-6.081689834590001|145.391998291     |5282    |10.0    |U  |Pacific/Port_Moresby|airport|OurAirports|
|2         |Madang Airport                             |Madang      |Papua New Guinea|MAG |AYMD|-5.20707988739    |145.789001465     |20      |10.0    |U  |Pacific/Port_Moresby|airport|OurAirports|
|3        

In [6]:
# En OpenFlights los faltantes pueden venir como NULL, cadena vacía, '\N', '-' o 'N/A'.
# Los contamos a todos para tener un panorama real de la calidad de cada columna.
MARCADORES = ["", "\\N", "-", "N/A", "NA"]

def reporte_nulos(df, titulo):
    total = df.count()
    filas = []
    for c in df.columns:
        col_str = F.trim(F.col(c).cast("string"))
        nulos = df.filter(F.col(c).isNull() | col_str.isin(MARCADORES)).count()
        pct = round(100 * nulos / total, 1) if total else 0.0
        filas.append((c, nulos, pct))
    print("\n" + titulo + f"  (total filas: {total})")
    (spark.createDataFrame(filas, ["columna", "nulos", "porcentaje"])
          .orderBy(F.desc("nulos"))
          .show(100, truncate=False))

reporte_nulos(airports_raw, "NULOS EN airports_raw")



NULOS EN airports_raw  (total filas: 12668)


[Stage 49:>                                                         (0 + 2) / 2]

+-----------+-----+----------+
|columna    |nulos|porcentaje|
+-----------+-----+----------+
|iata       |5744 |45.3      |
|icao       |4507 |35.6      |
|tz_database|4282 |33.8      |
|type       |1651 |13.0      |
|source     |1651 |13.0      |
|timezone   |353  |2.8       |
|dst        |353  |2.8       |
|city       |49   |0.4       |
|longitude  |0    |0.0       |
|airport_id |0    |0.0       |
|altitude   |0    |0.0       |
|name       |0    |0.0       |
|country    |0    |0.0       |
|latitude   |0    |0.0       |
+-----------+-----+----------+



In [7]:
# 'type' nos dice qué clase de registro es: la tabla mezcla aeropuertos con
# estaciones de tren, puertos, etc. Esto justifica quedarnos solo con type='airport'.
print("Distribución de 'type':")
(airports_raw
    .groupBy("type").count()
    .orderBy(F.desc("count"))
    .show(truncate=False))

print("Distribución de 'source':")
(airports_raw
    .groupBy("source").count()
    .orderBy(F.desc("count"))
    .show(truncate=False))

# Ejemplos concretos de cada tipo que NO es aeropuerto (para mostrar en el informe)
for t in ["station", "port", "unknown"]:
    print(f"\nEjemplos type = {t}:")
    (airports_raw
        .filter(F.col("type") == t)
        .select("airport_id", "name", "city", "country", "iata", "type", "source")
        .show(3, truncate=False))



Distribución de 'type':
+-------+-----+
|type   |count|
+-------+-----+
|airport|8264 |
|null   |1651 |
|station|1332 |
|unknown|1320 |
|port   |101  |
+-------+-----+

Distribución de 'source':
+-----------+-----+
|source     |count|
+-----------+-----+
|OurAirports|7698 |
|User       |3307 |
|null       |1651 |
|Legacy     |12   |
+-----------+-----+


Ejemplos type = station:
+----------+---------------------------+------------+-------------+----+-------+------+
|airport_id|name                       |city        |country      |iata|type   |source|
+----------+---------------------------+------------+-------------+----+-------+------+
|4035      |Cologne Railway            |Cologne     |Germany      |QKL |station|User  |
|4036      |Stuttgart Railway Station  |Stuttgart   |Germany      |ZWS |station|User  |
|6439      |New Rochelle Amtrak Station|New Rochelle|United States|null|station|User  |
+----------+---------------------------+------------+-------------+----+-------+------+
on

In [8]:

total = airports_raw.count()
distintos = airports_raw.select("airport_id").distinct().count()
print("airport_id duplicados:", total - distintos)              # esperado: 0
print("Filas exactas duplicadas:", total - airports_raw.dropDuplicates().count())
print("airport_id nulos:", airports_raw.filter(F.col("airport_id").isNull()).count())

# Si hubiera duplicados, este show los mostraría:
(airports_raw
    .groupBy("airport_id").count()
    .filter(F.col("count") > 1)
    .show(20, truncate=False))


airport_id duplicados: 0
Filas exactas duplicadas: 0
airport_id nulos: 0
+----------+-----+
|airport_id|count|
+----------+-----+
+----------+-----+



In [9]:
# Coordenadas fuera de rango geográfico válido
fuera_rango = airports_raw.filter(
    ~(F.col("latitude").between(-90, 90) & F.col("longitude").between(-180, 180))
).count()
print("Aeropuertos con coordenadas fuera de rango:", fuera_rango)   # esperado: 0

# Formato de códigos: IATA = 3 alfanum, ICAO = 4 alfanum (en mayúsculas).
# Cuenta cuántos NO cumplen (incluye los nulos/vacíos), para justificar que muchos
# faltan legítimamente y no se deben imputar.
print("IATA que no cumplen ^[A-Z0-9]{3}$:",
      airports_raw.filter(~F.upper(F.col("iata")).rlike("^[A-Z0-9]{3}$")).count())
print("ICAO que no cumplen ^[A-Z0-9]{4}$:",
      airports_raw.filter(~F.upper(F.col("icao")).rlike("^[A-Z0-9]{4}$")).count())



Aeropuertos con coordenadas fuera de rango: 0
IATA que no cumplen ^[A-Z0-9]{3}$: 5
ICAO que no cumplen ^[A-Z0-9]{4}$: 18


In [10]:
solo_aeropuertos = airports_raw.filter(F.col("type") == "airport")
print("Filas totales:", airports_raw.count())
print("Filas con type = 'airport':", solo_aeropuertos.count())
print("Filas descartadas (no son aeropuertos):",
      airports_raw.count() - solo_aeropuertos.count())

# Verificamos que tras el filtro la clave sigue siendo única
print("airport_id únicos tras filtro:",
      solo_aeropuertos.select("airport_id").distinct().count())



Filas totales: 12668
Filas con type = 'airport': 8264
Filas descartadas (no son aeropuertos): 4404
airport_id únicos tras filtro: 8264


# Análisis exploratorio airlines.csv

In [12]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("eda_openflights_airlines")
    .enableHiveSupport()
    .getOrCreate()
)

RAW_PATH = "/Obligatorio/landing/openflights/airlines.csv"

airlines_schema = T.StructType([
    T.StructField("airline_id", T.IntegerType(), True),
    T.StructField("name",       T.StringType(),  True),
    T.StructField("alias",      T.StringType(),  True),
    T.StructField("iata",       T.StringType(),  True),
    T.StructField("icao",       T.StringType(),  True),
    T.StructField("callsign",   T.StringType(),  True),
    T.StructField("country",    T.StringType(),  True),
    T.StructField("active",     T.StringType(),  True),
])

airlines_raw = (
    spark.read
    .option("header", True)
    .schema(airlines_schema)
    .csv(RAW_PATH)
)



In [13]:
print("Filas:", airlines_raw.count())
print("Columnas:", len(airlines_raw.columns))
print("Nombres de columnas:", airlines_raw.columns)

airlines_raw.printSchema()
airlines_raw.show(5, truncate=False)

Filas: 6162
Columnas: 8
Nombres de columnas: ['airline_id', 'name', 'alias', 'iata', 'icao', 'callsign', 'country', 'active']
root
 |-- airline_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- alias: string (nullable = true)
 |-- iata: string (nullable = true)
 |-- icao: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- country: string (nullable = true)
 |-- active: string (nullable = true)

+----------+--------------------------------------------+-----+----+----+--------+--------------+------+
|airline_id|name                                        |alias|iata|icao|callsign|country       |active|
+----------+--------------------------------------------+-----+----+----+--------+--------------+------+
|-1        |Unknown                                     |null |-   |N/A |null    |null          |Y     |
|1         |Private flight                              |null |-   |N/A |null    |null          |Y     |
|2         |135 Airways               

In [14]:
# En airlines los faltantes aparecen como vacío, '\N', '-' (en iata) y 'N/A' (en icao).
MARCADORES = ["", "\\N", "-", "N/A", "NA"]

def reporte_nulos(df, titulo):
    total = df.count()
    filas = []
    for c in df.columns:
        col_str = F.trim(F.col(c).cast("string"))
        nulos = df.filter(F.col(c).isNull() | col_str.isin(MARCADORES)).count()
        pct = round(100 * nulos / total, 1) if total else 0.0
        filas.append((c, nulos, pct))
    print("\n" + titulo + f"  (total filas: {total})")
    (spark.createDataFrame(filas, ["columna", "nulos", "porcentaje"])
          .orderBy(F.desc("nulos"))
          .show(100, truncate=False))

reporte_nulos(airlines_raw, "NULOS EN airlines_raw")



NULOS EN airlines_raw  (total filas: 6162)
+----------+-----+----------+
|columna   |nulos|porcentaje|
+----------+-----+----------+
|alias     |5983 |97.1      |
|iata      |4630 |75.1      |
|callsign  |822  |13.3      |
|icao      |275  |4.5       |
|country   |18   |0.3       |
|airline_id|0    |0.0       |
|active    |0    |0.0       |
|name      |0    |0.0       |
+----------+-----+----------+



In [15]:
# 'active' debería ser binario (Y/N) pero conviene chequear que no haya variantes
# de mayúsculas/minúsculas u otros valores antes de convertir a booleano.
print("Distribución de 'active':")
(airlines_raw
    .groupBy("active").count()
    .orderBy(F.desc("count"))
    .show(truncate=False))



Distribución de 'active':
+------+-----+
|active|count|
+------+-----+
|N     |4906 |
|Y     |1255 |
|n     |1    |
+------+-----+



In [16]:
total = airlines_raw.count()
distintos = airlines_raw.select("airline_id").distinct().count()
print("airline_id duplicados:", total - distintos)              # esperado: 0
print("Filas exactas duplicadas:", total - airlines_raw.dropDuplicates().count())
print("airline_id nulos:", airlines_raw.filter(F.col("airline_id").isNull()).count())

(airlines_raw
    .groupBy("airline_id").count()
    .filter(F.col("count") > 1)
    .show(20, truncate=False))


airline_id duplicados: 0
Filas exactas duplicadas: 0
airline_id nulos: 0
+----------+-----+
|airline_id|count|
+----------+-----+
+----------+-----+



In [17]:
# La fila 'Unknown' usa airline_id = -1: registro centinela, no una aerolínea real.
print("Filas con airline_id = -1 (Unknown):",
      airlines_raw.filter(F.col("airline_id") == -1).count())
airlines_raw.filter(F.col("airline_id") == -1).show(truncate=False)

# Formato esperado: IATA = 2 alfanum, ICAO = 3 alfanum. Cuenta los que NO cumplen
# (incluye nulos/vacíos) para mostrar que muchos faltan legítimamente.
print("IATA que no cumplen ^[A-Z0-9]{2}$:",
      airlines_raw.filter(~F.upper(F.col("iata")).rlike("^[A-Z0-9]{2}$")).count())
print("ICAO que no cumplen ^[A-Z0-9]{3}$:",
      airlines_raw.filter(~F.upper(F.col("icao")).rlike("^[A-Z0-9]{3}$")).count())




Filas con airline_id = -1 (Unknown): 1
+----------+-------+-----+----+----+--------+-------+------+
|airline_id|name   |alias|iata|icao|callsign|country|active|
+----------+-------+-----+----+----+--------+-------+------+
|-1        |Unknown|null |-   |N/A |null    |null   |Y     |
+----------+-------+-----+----+----+--------+-------+------+

IATA que no cumplen ^[A-Z0-9]{2}$: 18
ICAO que no cumplen ^[A-Z0-9]{3}$: 15


In [18]:
total = airlines_raw.count()

# 'alias' casi siempre está vacío: justifica considerarla poco útil.
alias_no_nulo = airlines_raw.filter(
    F.col("alias").isNotNull() & (F.trim(F.col("alias")) != "")
).count()
print(f"alias con valor: {alias_no_nulo} de {total} "
      f"({round(100*alias_no_nulo/total,1)}%)")

# Cantidad de países distintos representados
print("Países distintos:",
      airlines_raw.select("country")
                  .filter(F.col("country").isNotNull() & (F.trim(F.col("country")) != ""))
                  .distinct().count())



alias con valor: 179 de 6162 (2.9%)
Países distintos: 276


# Análisis exploratorio de routes.csv

In [19]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("eda_openflights_routes")
    .enableHiveSupport()
    .getOrCreate()
)

RAW_BASE = "/Obligatorio/landing/openflights"

routes_schema = T.StructType([
    T.StructField("airline",           T.StringType(),  True),
    T.StructField("airline_id",        T.IntegerType(), True),
    T.StructField("source_airport",    T.StringType(),  True),
    T.StructField("source_airport_id", T.IntegerType(), True),
    T.StructField("dest_airport",      T.StringType(),  True),
    T.StructField("dest_airport_id",   T.IntegerType(), True),
    T.StructField("codeshare",         T.StringType(),  True),
    T.StructField("stops",             T.IntegerType(), True),
    T.StructField("equipment",         T.StringType(),  True),
])

routes_raw = (
    spark.read
    .option("header", True)
    .schema(routes_schema)
    .csv(f"{RAW_BASE}/routes.csv")
)


2026-06-20T19:47:23,736 WARN [Thread-4] org.apache.spark.sql.SparkSession - Using an existing Spark session; only runtime SQL configurations will take effect.


In [20]:
print("Filas:", routes_raw.count())
print("Columnas:", len(routes_raw.columns))
print("Nombres de columnas:", routes_raw.columns)

routes_raw.printSchema()
routes_raw.show(5, truncate=False)


Filas: 67663
Columnas: 9
Nombres de columnas: ['airline', 'airline_id', 'source_airport', 'source_airport_id', 'dest_airport', 'dest_airport_id', 'codeshare', 'stops', 'equipment']
root
 |-- airline: string (nullable = true)
 |-- airline_id: integer (nullable = true)
 |-- source_airport: string (nullable = true)
 |-- source_airport_id: integer (nullable = true)
 |-- dest_airport: string (nullable = true)
 |-- dest_airport_id: integer (nullable = true)
 |-- codeshare: string (nullable = true)
 |-- stops: integer (nullable = true)
 |-- equipment: string (nullable = true)

+-------+----------+--------------+-----------------+------------+---------------+---------+-----+---------+
|airline|airline_id|source_airport|source_airport_id|dest_airport|dest_airport_id|codeshare|stops|equipment|
+-------+----------+--------------+-----------------+------------+---------------+---------+-----+---------+
|2B     |410       |AER           |2965             |KZN         |2990           |null     |0   

In [21]:
MARCADORES = ["", "\\N", "-", "N/A", "NA"]

def reporte_nulos(df, titulo):
    total = df.count()
    filas = []
    for c in df.columns:
        col_str = F.trim(F.col(c).cast("string"))
        nulos = df.filter(F.col(c).isNull() | col_str.isin(MARCADORES)).count()
        pct = round(100 * nulos / total, 1) if total else 0.0
        filas.append((c, nulos, pct))
    print("\n" + titulo + f"  (total filas: {total})")
    (spark.createDataFrame(filas, ["columna", "nulos", "porcentaje"])
          .orderBy(F.desc("nulos"))
          .show(100, truncate=False))

reporte_nulos(routes_raw, "NULOS EN routes_raw")




NULOS EN routes_raw  (total filas: 67663)
+-----------------+-----+----------+
|columna          |nulos|porcentaje|
+-----------------+-----+----------+
|codeshare        |53066|78.4      |
|airline_id       |479  |0.7       |
|dest_airport_id  |221  |0.3       |
|source_airport_id|220  |0.3       |
|equipment        |18   |0.0       |
|airline          |0    |0.0       |
|dest_airport     |0    |0.0       |
|source_airport   |0    |0.0       |
|stops            |0    |0.0       |
+-----------------+-----+----------+



In [22]:
# routes no trae un identificador propio. La clave candidata es la combinación
# (airline_id, source_airport_id, dest_airport_id, equipment).
clave = ["airline_id", "source_airport_id", "dest_airport_id", "equipment"]

total = routes_raw.count()
print("Filas:", total)
print("Filas exactas duplicadas:", total - routes_raw.dropDuplicates().count())
print("Duplicados por clave compuesta:", total - routes_raw.dropDuplicates(clave).count())

# Ejemplos de rutas duplicadas por la clave compuesta
(routes_raw
    .groupBy(*clave).count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .show(20, truncate=False))



Filas: 67663
Filas exactas duplicadas: 0
Duplicados por clave compuesta: 64
+----------+-----------------+---------------+---------+-----+
|airline_id|source_airport_id|dest_airport_id|equipment|count|
+----------+-----------------+---------------+---------+-----+
|921       |null             |null           |BH2      |13   |
|921       |null             |5448           |BH2      |7    |
|921       |5448             |null           |BH2      |7    |
|921       |null             |8628           |BH2      |6    |
|921       |8628             |null           |BH2      |6    |
|921       |7642             |null           |BH2      |4    |
|921       |null             |7642           |BH2      |4    |
|921       |5442             |null           |BH2      |3    |
|921       |null             |5442           |BH2      |3    |
|3969      |null             |2923           |AN4      |3    |
|3969      |2923             |null           |AN4      |3    |
|692       |null             |null        

In [23]:
# stops: casi todas deberían ser directas (0). Justifica derivar is_direct.
print("Distribución de 'stops':")
routes_raw.groupBy("stops").count().orderBy("stops").show()

# codeshare: solo tiene valor cuando la ruta es código compartido (muchos nulos = normal).
con_codeshare = routes_raw.filter(
    F.col("codeshare").isNotNull() & (F.trim(F.col("codeshare")) != "")
).count()
total = routes_raw.count()
print(f"Rutas marcadas como codeshare: {con_codeshare} de {total} "
      f"({round(100*con_codeshare/total,1)}%)")

# Rutas inconsistentes: origen == destino (deben descartarse)
print("Rutas con origen == destino:",
      routes_raw.filter(F.col("source_airport_id") == F.col("dest_airport_id")).count())


Distribución de 'stops':
+-----+-----+
|stops|count|
+-----+-----+
|    0|67652|
|    1|   11|
+-----+-----+

Rutas marcadas como codeshare: 14597 de 67663 (21.6%)
Rutas con origen == destino: 1


In [24]:
# Cargamos las otras dos tablas para verificar que cada ruta apunte a un aeropuerto
# y una aerolínea existentes. Esto justifica el filtro de integridad en el refinamiento.
airports_ids = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{RAW_BASE}/airports.csv")
    .select("airport_id")
)
airlines_ids = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{RAW_BASE}/airlines.csv")
    .select("airline_id")
)

# source_airport_id inexistente en airports
src_huerfanas = routes_raw.join(
    airports_ids.withColumnRenamed("airport_id", "source_airport_id"),
    "source_airport_id", "left_anti"
).count()

# dest_airport_id inexistente en airports
dst_huerfanas = routes_raw.join(
    airports_ids.withColumnRenamed("airport_id", "dest_airport_id"),
    "dest_airport_id", "left_anti"
).count()

# airline_id inexistente en airlines
al_huerfanas = routes_raw.join(
    airlines_ids, "airline_id", "left_anti"
).count()

print("Rutas con source_airport_id inexistente:", src_huerfanas)
print("Rutas con dest_airport_id inexistente:", dst_huerfanas)
print("Rutas con airline_id inexistente:", al_huerfanas)



Rutas con source_airport_id inexistente: 220
Rutas con dest_airport_id inexistente: 221
Rutas con airline_id inexistente: 479
